In [2]:
import pandas as pd
import pvlib
from pathlib import Path

# -------------------------------------------------------------
# CHANGE THIS TO POINT TO ONE OF YOUR FORMATTED WEATHER FILES
# -------------------------------------------------------------
csv_path = Path("/Users/akshathkandadai/Desktop/Solar Data/formatted/118/118_248116_33.53_-115.78_1999.csv")


# -------------------------------------------------------------
# LOAD WEATHER DATA
# -------------------------------------------------------------
df = pd.read_csv(csv_path, parse_dates=["datetime"])
df = df.set_index("datetime")

# Extract metadata from filename
bus, locid, lat, lon, year = csv_path.stem.split("_")
lat = float(lat)
lon = float(lon)

# -------------------------------------------------------------
# DEFINE LOCATION + PV SYSTEM CONFIG (RTS-GMLC assumptions)
# -------------------------------------------------------------
location = pvlib.location.Location(latitude=lat, longitude=lon, tz="UTC")

DC_CAPACITY = 3000   # watts (DC side)
AC_CAPACITY = 2500   # watts (AC inverter side) → ILR = 1.2

system = pvlib.pvsystem.PVSystem(
    surface_tilt=lat,             # fixed tilt = latitude
    surface_azimuth=180,          # south facing
    module_parameters={"pdc0": DC_CAPACITY},
    inverter_parameters={"pdc0": AC_CAPACITY},
)

# If surface_albedo isn't present, default to 0.2
df["surface_albedo"] = df.get("surface_albedo", 0.2)

# Required fields for PVWatts
weather = df[["dni", "ghi", "dhi", "air_temperature", "wind_speed", "surface_albedo"]]

# -------------------------------------------------------------
# RUN PV MODEL: PVWatts (fast + simple)
# -------------------------------------------------------------
mc = pvlib.modelchain.ModelChain.with_pvwatts(system, location)
mc.run_model(weather)

ac_power = mc.results.ac   # AC power output (watts)

# -------------------------------------------------------------
# NORMALIZE POWER → (0 to 1)
# -------------------------------------------------------------
df["profile"] = (ac_power / ac_power.max()).clip(lower=0, upper=1)

# -------------------------------------------------------------
# SAVE OUTPUT
# -------------------------------------------------------------
out_path = csv_path.parent / f"{csv_path.stem}_PROFILE.csv"
df[["profile"]].to_csv(out_path)

print("✅ Done! Generated profile:")
print(df[["profile"]].head())

print(f"\n📁 Saved to: {out_path}")


KeyError: "['dni', 'ghi', 'dhi', 'air_temperature', 'wind_speed'] not in index"